<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/Audio_RAVDESS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle librosa soundfile -q

In [2]:
import kagglehub
import os

dataset_path = kagglehub.dataset_download(
    "uwrfkaggler/ravdess-emotional-speech-audio"
)

print("Dataset path:", dataset_path)
print("Folders:", os.listdir(dataset_path))

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Dataset path: /kaggle/input/ravdess-emotional-speech-audio
Folders: ['Actor_02', 'Actor_17', 'Actor_05', 'Actor_16', 'Actor_21', 'Actor_01', 'Actor_11', 'Actor_20', 'Actor_08', 'Actor_15', 'Actor_06', 'Actor_12', 'Actor_23', 'Actor_24', 'Actor_22', 'Actor_04', 'Actor_19', 'Actor_10', 'Actor_09', 'audio_speech_actors_01-24', 'Actor_14', 'Actor_03', 'Actor_13', 'Actor_18', 'Actor_07']


In [3]:
for root, dirs, files in os.walk(dataset_path):
    print("Current path:", root)
    print("Number of files:", len(files))
    print("-" * 30)

Current path: /kaggle/input/ravdess-emotional-speech-audio
Number of files: 0
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_02
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_17
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_05
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_16
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_21
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_01
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio/Actor_11
Number of files: 60
------------------------------
Current path: /kaggle/input/ravdess-emotional-speech-audio

In [4]:
audio_files = []
labels = []

emotion_map = {
    "01": "neutral",
    "03": "happy",
    "04": "sad",
    "05": "angry"
}

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".wav"):
            parts = file.split("-")
            emotion_code = parts[2]

            if emotion_code in emotion_map:
                audio_files.append(os.path.join(root, file))
                labels.append(emotion_map[emotion_code])

print("Total samples:", len(audio_files))

Total samples: 1344


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
encoded_labels = le.fit_transform(labels)

print(le.classes_)

['angry' 'happy' 'neutral' 'sad']


In [6]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

if device == "cuda":
    print(torch.cuda.get_device_name(0))

Using: cuda
Tesla T4


In [7]:
import librosa
import numpy as np

def extract_features(file_path, sr=22050, duration=3):
    signal, sample_rate = librosa.load(file_path, sr=sr, mono=True)

    target_len = sr * duration

    if len(signal) < target_len:
        signal = np.pad(signal, (0, target_len - len(signal)))
    else:
        signal = signal[:target_len]

    mel = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    return mel_db

In [8]:
X = []
y = []

for file, label in zip(audio_files, encoded_labels):
    feat = extract_features(file)
    X.append(feat)
    y.append(label)

X = np.array(X)
y = np.array(y)

# Add channel dimension
X = np.expand_dims(X, axis=1)

print("X shape:", X.shape)

X shape: (1344, 1, 128, 130)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

In [11]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=16
)

In [15]:
import torch.nn as nn

class AudioCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),   # Corrected from 128 * 16 * 8 to 128 * 16 * 16
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [16]:
model = AudioCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [17]:
from tqdm import tqdm

epochs = 15

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for Xb, yb in tqdm(train_loader):
        Xb, yb = Xb.to(device), yb.to(device)

        out = model(Xb)
        loss = criterion(out, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss:", total_loss / len(train_loader))

100%|██████████| 68/68 [00:01<00:00, 42.31it/s]


Epoch 1 Loss: 3.359205021577723


100%|██████████| 68/68 [00:00<00:00, 71.85it/s]


Epoch 2 Loss: 1.183969844790066


100%|██████████| 68/68 [00:00<00:00, 71.59it/s]


...

Epoch 13 Loss: 0.6813116884406876


100%|██████████| 68/68 [00:00<00:00, 71.10it/s]


Epoch 14 Loss: 0.6294275149703026


100%|██████████| 68/68 [00:00<00:00, 71.13it/s]

Epoch 15 Loss: 0.5849254647379413


In [18]:
from sklearn.metrics import accuracy_score

model.eval()
preds = []
true = []

with torch.no_grad():
    for Xb, yb in val_loader:
        Xb = Xb.to(device)

        out = model(Xb)
        pred = torch.argmax(out, dim=1)

        preds.extend(pred.cpu().numpy())
        true.extend(yb.numpy())

acc = accuracy_score(true, preds)
print("Validation Accuracy:", acc)

Validation Accuracy: 0.7992565055762082


In [19]:
torch.save(model.state_dict(), "/content/audio_model.pth")

In [20]:
import json

label_map = {int(i): label for i, label in enumerate(le.classes_)}

with open("/content/audio_label_map.json", "w") as f:
    json.dump(label_map, f)

print("Label map saved!")

Label map saved!
